# Colab GPU training — Real-Time Threat Detection

Reproduces paper Table II (YOLOv8, 50 & 100 epochs) on a Colab GPU.

**Before running:** Runtime → Change runtime type → **GPU** (T4 OK; L4/A100 faster for larger batch).

Mount Drive early (cell below) so checkpoints survive disconnects.

In [ ]:
# <<< EDIT THIS after you push to GitHub >>>
REPO_URL = "https://github.com/jon44ai-svg/realtime-threat-detection.git"
BRANCH = "feat/initial-workspace"

import os, pathlib
ROOT = pathlib.Path("/content/realtime-threat-detection")
if not ROOT.exists():
    !git clone --branch {BRANCH} {REPO_URL} {ROOT}
%cd {ROOT}
!pip install -q ultralytics opencv-python-headless pyyaml matplotlib seaborn kaggle python-dotenv pillow
!pip install -q -e .
import sys, torch
sys.path.insert(0, str(ROOT / "src"))
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## Persist to Google Drive (run early)

Creates `MyDrive/realtime-threat-detection/colab_runs/<timestamp>_<GPU>/` and copies named checkpoints after each train.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil, datetime, json, torch

drive.mount("/content/drive")

GPU = (torch.cuda.get_device_name(0).replace(" ", "_") if torch.cuda.is_available() else "cpu")
RUN_ID = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S") + "_" + GPU
DRIVE_ROOT = Path("/content/drive/MyDrive/realtime-threat-detection") / "colab_runs" / RUN_ID
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive run folder:", DRIVE_ROOT)

def sync_artifacts(tag: str = "checkpoint"):
    """Copy weights + eval outputs to Drive with meaningful names."""
    stamp = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
    dest = DRIVE_ROOT / f"{stamp}_{tag}"
    dest.mkdir(parents=True, exist_ok=True)
    meta = {"tag": tag, "run_id": RUN_ID, "gpu": GPU, "utc": stamp, "repo": str(ROOT)}
    copied = []
    for rel in [
        "runs/detect/train_50/weights/best.pt",
        "runs/detect/train_50/weights/last.pt",
        "runs/detect/train_100/weights/best.pt",
        "runs/detect/train_100/weights/last.pt",
        "runs/detect/train_smoke/weights/best.pt",
    ]:
        src = ROOT / rel
        if not src.exists():
            continue
        if "train_50" in rel:
            epochs = "50"
        elif "train_100" in rel:
            epochs = "100"
        else:
            epochs = "smoke"
        name = f"yolov8n_epochs{epochs}_{src.stem}_{GPU}_{stamp}.pt"
        shutil.copy2(src, dest / name)
        copied.append(name)
    for pattern in [
        "runs/**/results.csv",
        "runs/**/metrics_vs_paper.json",
        "runs/**/confusion_matrix*.png",
        "runs/**/BoxPR_curve.png",
        "runs/**/args.yaml",
    ]:
        for src in ROOT.glob(pattern):
            out = dest / src.relative_to(ROOT)
            out.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, out)
            copied.append(str(src.relative_to(ROOT)))
    (dest / "manifest.json").write_text(json.dumps({**meta, "copied": copied}, indent=2))
    print(f"Synced {len(copied)} files → {dest}")
    return dest

sync_artifacts("startup")

## Kaggle credentials
Upload `kaggle.json` via the Files sidebar, or paste username/key below.

In [ ]:
from pathlib import Path
import json, os

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
uploaded = Path("/content/kaggle.json")
if uploaded.exists():
    dest = kaggle_dir / "kaggle.json"
    dest.write_text(uploaded.read_text())
    os.chmod(dest, 0o600)
    print("Using /content/kaggle.json")
else:
    USERNAME = ""
    KEY = ""
    if not KEY:
        raise SystemExit("Upload /content/kaggle.json or set KEY in this cell")
    (kaggle_dir / "kaggle.json").write_text(json.dumps({"username": USERNAME, "key": KEY}))
    os.chmod(kaggle_dir / "kaggle.json", 0o600)
    print("Wrote ~/.kaggle/kaggle.json from cell vars")

In [ ]:
import sys
sys.path.insert(0, str(ROOT / "src"))
from threat_detection.data.download import download_dataset, write_data_yaml

ds = download_dataset(ROOT / "data" / "raw")
write_data_yaml(ds, ROOT / "configs" / "data.yaml")
print("dataset:", ds)

## Train 50 then 100 epochs (paper Table II)

Each train cell ends with `sync_artifacts(...)` so Drive has the checkpoint even if the runtime dies next.

In [ ]:
!python scripts/train.py --epochs 50 --device 0
!python scripts/evaluate.py --weights runs/detect/train_50/weights/best.pt --epochs 50
sync_artifacts("after_epochs50")

In [ ]:
!python scripts/train.py --epochs 100 --device 0
!python scripts/evaluate.py --weights runs/detect/train_100/weights/best.pt --epochs 100
sync_artifacts("after_epochs100")

## Manual re-sync

If training finished but you forgot sync, run:

In [ ]:
sync_artifacts("manual_resync")
print("Also available under:", DRIVE_ROOT)